# ทดลองโมเดลจำแนกตัวอักษรไทย

เราจะเริ่มจากเทียบโมเดล 4 แบบ เลือกตัวที่ทำได้ดี แล้วค่อยปรับวิธีฝึกและจูนค่า ผลสุดท้ายคือ config กับไฟล์โมเดลที่นำไปใช้ต่อได้

ทุกการทดลองใช้ train/validation ชุดเดียวกัน เลือกโมเดลจาก **macro F1** ซึ่งให้น้ำหนักแต่ละคลาสเท่ากัน และดู accuracy ประกอบ

เริ่มจากเลือก kernel `.venv` แล้วแก้ค่าด้านล่าง ถ้าใช้ข้อมูลจาก `.env` เดิมอยู่แล้ว ปล่อยช่อง path ว่างได้ จากนั้นกด **Run All**


## 1. ตั้งค่าการทดลอง

| โหมด | เทียบโมเดล | จูน Optuna | เทรนยืนยัน |
| --- | --- | --- | --- |
| `smoke` | ภาพจำลอง, 2 epochs | 2 trials | 2 epochs, 1 seed |
| `quick` | 6 epochs | 8 trials × 6 epochs | 20 epochs, 1 seed |
| `full` | 12 epochs | 12 trials × 8 epochs | 30 epochs, 2 seeds |

ใช้ `quick` เริ่มทดลองด้วยข้อมูลจริงที่ดาวน์โหลดมา ส่วน `full` เพิ่มงบการฝึกและอาจใช้เวลาหลายชั่วโมง หากหน่วยความจำ GPU ไม่พอ ให้ลด `BATCH_SIZE` แล้วเทียบทุกโมเดลใหม่ด้วยค่าเดียวกัน


## แผนแบ่งการทดลองให้ทีม

**ก่อนเริ่มทุกคน:** ใช้ `data/splits/` ชุดกลางเดียวกัน, `SEED = 42`, `IMAGE_SIZE = 224`, `PAD_VALUE = 255`, validation preprocessing เดียวกัน และส่ง `run_receipt.json` + `config.json` + `metrics.json` กลับทีม. ห้ามสร้าง split ใหม่หรือเทียบคะแนนข้าม `MODE`.

### รอบที่ 1 — เลือก architecture (เปลี่ยนได้เฉพาะ architecture)

ให้คนรับผิดชอบแต่ละคนตั้ง `ARCHITECTURES` เหลือเฉพาะโมเดลของตน แล้วรันถึงส่วน **6. เทียบโมเดล**. `custom_cnn` ถูกเพิ่มอัตโนมัติ จึงใช้ผลจากสมาชิกคนเดียวเป็น baseline กลางได้. ทุกคนใช้ config ฐานเดียวกัน: pretrained `True` สำหรับ transfer model, freeze 2 epochs, head LR `0.001`, fine-tune LR `0.0001`, weight decay `0.0001`, dropout `0.3`, augmentation/class weights ปิด.

| งาน | ผู้รัน | ค่า `ARCHITECTURES` | เป้าหมาย |
| --- | --- | --- | --- |
| E0 baseline | 1 คน | `[]` | `custom_cnn` สำหรับเป็นค่าตั้งต้น |
| E1a | เพื่อน A | `["resnet18"]` | transfer-learning baseline ที่เสถียร |
| E1b | เพื่อน B | `["efficientnet_b0"]` | ตัวเลือกสมดุล accuracy/compute |
| E1c | เพื่อน C | `["mobilenetv3_large_100"]` | โมเดลเบา เป็น backup/demo |
| E1d | เพื่อน D | `["convnext_tiny"]` | modern CNN รุ่นใหม่ |
| E1e | เพื่อน E | `["efficientnetv2_rw_s"]` | efficient CNN รุ่นใหม่ |
| E1f | เพื่อน F | `["regnety_032"]` | CNN คนละ family สำหรับเทียบ |
| E1g | เพื่อน G | `["swin_tiny_patch4_window7_224"]` | Vision Transformer ตัวแทน |

ใช้ `MODE = "quick"` เพื่อคัดกรองบนเครื่องของแต่ละคน (6 epochs) แล้วนำอันดับ 1–2 ไปรัน `MODE = "full"` ด้วย config เดิมเพื่อยืนยันก่อนเลือกผู้ชนะ. ถ้า GPU ไม่พอ ให้ลด `BATCH_SIZE` เช่น 32 → 16 **สำหรับทุกรุ่นในรอบเดียวกัน** และบันทึกค่านั้นใน config; อย่านำผลคนละ batch size มาอยู่ leaderboard เดียวกัน.

### รอบที่ 2 — ปรับ config ของผู้ชนะ (เปลี่ยนทีละปัจจัย)

1. รันส่วน **7. augmentation และ class weights** ของ architecture ที่ชนะ: เปรียบเทียบ control → augmentation → class weights. เก็บการเปลี่ยนแปลงเมื่อ macro F1 ดีขึ้นและไม่ทำให้ minority recall แย่ลง.
2. จากนั้นรันส่วน **8. Optuna** ของ recipe ที่ชนะเท่านั้น: ค้นหา head LR, fine-tune LR, weight decay และ dropout ตามช่วงที่ notebook กำหนด. ห้ามจูน architecture, augmentation และ loss พร้อมกัน.
3. รันส่วน **9. เทรนยืนยัน**: นำ untuned control และ 2 Optuna trials ที่ดีที่สุดมาเทรนเต็ม budget; `full` จะยืนยันด้วย seed 42 และ 123. เลือกจากค่าเฉลี่ย macro F1, แล้วดู accuracy, minimum per-class recall และเวลาเทรนประกอบ.

### กติกาส่งผลและตัดสิน

ตั้ง `OUTPUT_ROOT` คนละโฟลเดอร์ถ้ารันพร้อมกัน เพื่อไม่ให้ชนกัน (เช่น `results/model_search_mint`). ผู้รวมผลรับเฉพาะ run ที่มี split hash เดียวกัน, config ครบ, และ checkpoint/receipt อยู่จริง. เลือก **winner 1 ตัว + backup 1 ตัว** จาก `leaderboard.csv`; โมเดลสุดท้ายต้องใช้ `export/best_config.json` และ `export/best_model.pt` ที่ notebook สร้าง ไม่คัดลอกค่าจากหน้าจอ.


In [1]:
from pathlib import Path
import sys

MODE = "quick"  # ใช้ข้อมูลจริง; เปลี่ยนเป็น full เมื่อพร้อมรันนาน
SEED = 42
DEVICE = "auto"  # เลือก CUDA, Apple MPS หรือ CPU ตามเครื่อง

# เพิ่มชื่อโมเดลจาก timm ในรายการนี้ได้; Custom CNN จะถูกเพิ่มให้เอง
ARCHITECTURES = [
    "resnet18",
    "efficientnet_b0",
    "mobilenetv3_large_100",
]

IMAGE_SIZE = 224
BATCH_SIZE = 32
PAD_VALUE = 255  # สีที่เติมขอบภาพ: 255 = ขาว, 0 = ดำ
EXPECTED_CLASSES = 72
LABEL_LEVEL = 2  # clean_32x32/<หมวด>/<คลาส>/<ภาพ>

# ปล่อยว่างเพื่อใช้ path ของโปรเจกต์และค่าจาก .env
REPO_PATH = ""
DATA_PATH = "data/raw/clean_32x32.zip"  # ระบบจะแตก ZIP ให้อัตโนมัติ
SPLIT_PATH = ""
OUTPUT_ROOT = ""
DOWNLOAD_IF_MISSING = True


### เตรียมโฟลเดอร์โปรเจกต์

Cell นี้หา `src/` และเลือกจำนวน epochs ตามโหมดที่ตั้งไว้ ปกติไม่ต้องแก้
ถ้าใช้ Colab ให้ clone หรือ upload ทั้ง repo แล้วตั้ง `REPO_PATH` ให้ตรง ส่วนผลและ split ควรเก็บบน Drive เพื่อไม่ให้หายตอนปิด session


In [2]:
if REPO_PATH:
    project_candidates = [Path(REPO_PATH).expanduser()]
else:
    project_candidates = [Path.cwd(), *Path.cwd().parents]

project_dir = None
for folder in project_candidates:
    if (folder / "src/train.py").is_file():
        project_dir = folder.resolve()
        break

if project_dir is None:
    raise FileNotFoundError("หา src/train.py ไม่เจอ ให้ตั้ง REPO_PATH เป็นโฟลเดอร์ repo")

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

training_budgets = {
    "smoke": {"benchmark": 2, "hpo_epochs": 2, "trials": 2, "final_epochs": 2, "seeds": (42,)},
    "quick": {"benchmark": 6, "hpo_epochs": 6, "trials": 8, "final_epochs": 20, "seeds": (42,)},
    "full": {"benchmark": 12, "hpo_epochs": 8, "trials": 12, "final_epochs": 30, "seeds": (42, 123)},
}
training_budget = training_budgets[MODE]

print("โฟลเดอร์โปรเจกต์:", project_dir)
print("โหมด:", MODE)
print("จำนวน epochs และ trials:", training_budget)


โฟลเดอร์โปรเจกต์: /Users/thawaree.kha/kmitl-deeplearn-project1
โหมด: quick
จำนวน epochs และ trials: {'benchmark': 6, 'hpo_epochs': 6, 'trials': 8, 'final_epochs': 20, 'seeds': (42,)}


## 2. เตรียมไลบรารี

ติดตั้งเฉพาะเมื่อยังมีไลบรารีไม่ครบ ครั้งแรกต้องต่ออินเทอร์เน็ต หากติดตั้งแล้ว import ไม่ผ่าน ให้ restart kernel แล้วรันใหม่


In [3]:
import importlib.util
import subprocess

required_packages = [
    "torch", "torchvision", "timm", "optuna", "numpy",
    "pandas", "sklearn", "matplotlib", "PIL", "gdown",
]
missing_packages = []
for package in required_packages:
    if importlib.util.find_spec(package) is None:
        missing_packages.append(package)

if missing_packages:
    print("กำลังติดตั้งไลบรารีที่ยังขาด:", missing_packages)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(project_dir / "requirements-training.txt")],
        check=True,
    )
else:
    print("ไลบรารีพร้อมแล้ว")


ไลบรารีพร้อมแล้ว


In [4]:
import json
import shutil
from dataclasses import asdict, replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image

from src.audit import DEFAULT_EXTENSIONS, audit_dataset, image_paths
from src.inference import Predictor
from src.paths import configured_data_dir, prepare_image_directory
from src.search import config_from_receipt, confirm, run_ablations, run_benchmark, tune
from src.split import prepare_split, read_rows
from src.train import TrainConfig, choose_device, make_transform, save_json

training_device = choose_device(DEVICE)
print("PyTorch:", torch.__version__)
print("อุปกรณ์ที่ใช้เทรน:", training_device)
if training_device.type == "cpu" and MODE != "smoke":
    print("รันบน CPU ได้ แต่การทดลองเต็มชุดจะช้ากว่า GPU มาก")


PyTorch: 2.14.0
อุปกรณ์ที่ใช้เทรน: mps


/Users/thawaree.kha/kmitl-deeplearn-project1/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## รายชื่อ architecture ที่รันได้ และชุดแบ่งงาน

Cell ถัดไปจะแสดง **รายชื่อ timm model ทั้งหมดที่มี pretrained weights ใน environment ปัจจุบัน** จึงเป็น source of truth ของเครื่องที่ใช้รันจริง. รายชื่ออาจต่างกันเล็กน้อยตามเวอร์ชัน `timm`; model ที่มีในตารางนี้เท่านั้นที่นำไปใส่ `ARCHITECTURES` ได้.

ไม่ควรรันทุกชื่อในรอบแรก: ใช้ 7 ตัวด้านล่างเพื่อให้ครอบคลุม CNN, efficient CNN และ transformer แล้วให้แต่ละคนคัดลอกเฉพาะชื่อที่ได้รับมอบหมายไปไว้ใน `ARCHITECTURES`. ทุกรุ่นใช้ input 224, batch size เดียวกัน, split เดียวกัน และ config ฐานเดียวกัน.

| งาน | Architecture | กลุ่ม | หมายเหตุ |
| --- | --- | --- | --- |
| E1a | `resnet18` | CNN baseline | เร็วและเป็น baseline transfer learning |
| E1b | `efficientnet_b0` | Efficient CNN | สมดุล accuracy/compute |
| E1c | `mobilenetv3_large_100` | Mobile CNN | ตัวสำรองสำหรับ demo/เครื่องช้า |
| E1d | `convnext_tiny` | Modern CNN | ตัวเลือกหลักรุ่นใหม่ |
| E1e | `efficientnetv2_rw_s` | Efficient CNN รุ่นใหม่ | เทียบกับ EfficientNet-B0 |
| E1f | `regnety_032` | CNN อีก family | ตรวจว่าผลไม่ได้ขึ้นกับ family เดียว |
| E1g | `swin_tiny_patch4_window7_224` | Vision Transformer | ทดลอง transformer เพียงหนึ่งตัวก่อน |

ตัวอย่างเพื่อน D รับ E1d: ตั้ง `ARCHITECTURES = ["convnext_tiny"]`, ตั้ง `OUTPUT_ROOT` เป็นโฟลเดอร์ชื่อของตน, แล้วรันถึงส่วน 6. ถ้าชื่อในตาราง recommended ไม่แสดงใน cell catalog ให้ใช้ชื่อที่ catalog แสดงแทน และแจ้งทีมก่อนเริ่มรัน.


In [5]:
import timm

# รายชื่อที่รันพร้อม pretrained weights ได้จริงใน timm เวอร์ชันของ kernel นี้
available_architectures = sorted(timm.list_models(pretrained=True))
architecture_catalog = pd.DataFrame({"architecture": available_architectures})
display(architecture_catalog)
print(f"timm {timm.__version__}: {len(architecture_catalog)} architectures with pretrained weights")

recommended_architectures = [
    "resnet18", "efficientnet_b0", "mobilenetv3_large_100",
    "convnext_tiny", "efficientnetv2_rw_s", "regnety_032",
    "swin_tiny_patch4_window7_224",
]
missing_recommendations = sorted(set(recommended_architectures) - set(available_architectures))
if missing_recommendations:
    print("ต้องเลือกชื่อทดแทนจาก catalog สำหรับ:", missing_recommendations)


,architecture
0,aimv2_1b_patch14_224.apple_pt
1,aimv2_1b_patch14_336.apple_pt
2,aimv2_1b_patch14_448.apple_pt
3,aimv2_3b_patch14_224.apple_pt
4,aimv2_3b_patch14_336.apple_pt
...,...
1732,xcit_tiny_24_p16_224.fb_in1k
1733,xcit_tiny_24_p16_384.fb_dist_in1k
1734,xcit_tiny_24_p8_224.fb_dist_in1k
1735,xcit_tiny_24_p8_224.fb_in1k


timm 1.0.29: 1737 architectures with pretrained weights
ต้องเลือกชื่อทดแทนจาก catalog สำหรับ: ['convnext_tiny', 'efficientnet_b0', 'efficientnetv2_rw_s', 'mobilenetv3_large_100', 'regnety_032', 'resnet18', 'swin_tiny_patch4_window7_224']


## 3. เตรียมภาพ

ใช้ภาพที่แตกไฟล์แล้วและแยกโฟลเดอร์ตามคลาส ถ้ายังไม่มีภาพ ระบบจะดาวน์โหลด public Drive folder จาก `.env` ให้ สำหรับ private Drive ให้ mount เองแล้วใส่ `DATA_PATH`

โหมด `smoke` จะสร้างภาพจำลองแยกไว้เพื่อทดสอบระบบ คะแนนของโหมดนี้ใช้วัดคุณภาพโมเดลไม่ได้


In [6]:
if OUTPUT_ROOT:
    results_dir = Path(OUTPUT_ROOT).expanduser().resolve()
else:
    results_dir = project_dir / "results/model_search"
results_dir.mkdir(parents=True, exist_ok=True)
if MODE == "smoke":
    data_dir = results_dir / "smoke_fixture/images"
    # เป็นข้อมูลทดสอบระบบเท่านั้น คะแนนจากภาพเหล่านี้ไม่มีความหมายด้านคุณภาพโมเดล
    rng = np.random.default_rng(42)
    for label in range(3):
        folder = data_dir / f"synthetic_{label}"
        folder.mkdir(parents=True, exist_ok=True)
        for index in range(10):
            path = folder / f"{index:03d}.png"
            if not path.exists():
                pixels = rng.integers(0, 40, (40, 32, 3), dtype=np.uint8)
                pixels[:, :, label] += 180
                Image.fromarray(pixels).save(path)
    split_dir = results_dir / "smoke_fixture/splits"
    EXPECTED_CLASSES = 3
    LABEL_LEVEL = 1
    IMAGE_SIZE = 32
    BATCH_SIZE = 8
    ARCHITECTURES = ["resnet18"]
else:
    if DATA_PATH:
        requested_data = Path(DATA_PATH).expanduser()
        data_dir = (requested_data if requested_data.is_absolute() else project_dir / requested_data).resolve()
    else:
        data_dir = configured_data_dir().resolve()

    if SPLIT_PATH:
        requested_split = Path(SPLIT_PATH).expanduser()
        split_dir = (requested_split if requested_split.is_absolute() else project_dir / requested_split).resolve()
    else:
        split_dir = project_dir / "data/splits/clean_32x32"
    data_dir = prepare_image_directory(data_dir, DEFAULT_EXTENSIONS)
    marker = data_dir / ".download_incomplete"
    if marker.exists():
        raise RuntimeError(f"Previous download incomplete: {data_dir}. Restore a complete cache before removing the marker.")
    if not list(image_paths(data_dir, DEFAULT_EXTENSIONS)):
        if not DOWNLOAD_IF_MISSING:
            raise FileNotFoundError("No images found. Set DATA_PATH to the extracted image folder.")
        if data_dir.exists() and any(data_dir.iterdir()):
            raise RuntimeError("DATA_DIR is non-empty but contains no images. Check for ZIP files or a wrong path.")
        # The downloader refuses non-empty directories; marker goes in only after failure.
        try:
            subprocess.run([sys.executable, "-m", "src.download_data", "--output-dir", str(data_dir)],
                           cwd=project_dir, check=True)
        except BaseException:
            data_dir.mkdir(parents=True, exist_ok=True)
            marker.touch()
            raise

print("โฟลเดอร์ภาพ:", data_dir)


โฟลเดอร์ภาพ: /Users/thawaree.kha/kmitl-deeplearn-project1/data/raw/clean_32x32


### ตรวจข้อมูลก่อนแบ่งชุด

ดูจำนวนภาพ จำนวนคลาส และไฟล์ที่อ่านไม่ได้ ถ้ามีไฟล์เสียหรือหา label ไม่ได้ จะหยุดตรงนี้ให้แก้ข้อมูลก่อน


In [7]:
audit_dir = results_dir / MODE / "audit"
audit_report = audit_dataset(
    data_dir=data_dir,
    output_dir=audit_dir,
    label_level=LABEL_LEVEL,
    extensions=DEFAULT_EXTENSIONS,
)

audit_summary = {
    "ภาพที่อ่านได้": audit_report["valid_images"],
    "จำนวนคลาส": audit_report["class_count"],
    "ไฟล์เสีย": audit_report["corrupt_images"],
    "ไฟล์ที่หา label ไม่ได้": audit_report["label_errors"],
    "กลุ่มภาพซ้ำ": audit_report["exact_duplicate_groups"],
}
display(pd.Series(audit_summary, name="ผลตรวจข้อมูล"))

if audit_report["corrupt_images"] or audit_report["label_errors"]:
    raise ValueError(f"พบข้อมูลที่ต้องแก้ ดู corrupt_images.csv และ label_errors.csv ใน {audit_dir}")


ภาพที่อ่านได้             278337
จำนวนคลาส                      5
ไฟล์เสีย                       0
ไฟล์ที่หา label ไม่ได้         0
กลุ่มภาพซ้ำ                 7893
Name: ผลตรวจข้อมูล, dtype: int64

## 4. แบ่ง train / validation

ตั้งเป้า 80/20 ในแต่ละคลาส ภาพที่ซ้ำกันต้องอยู่ฝั่งเดียวกัน สัดส่วนจริงจึงอาจคลาดเคลื่อนเล็กน้อย แต่ละคลาสต้องมีอย่างน้อย 2 กลุ่มภาพที่ไม่ซ้ำกัน

รันครั้งแรกจะบันทึกรายชื่อภาพไว้ ครั้งต่อไปใช้ชุดเดิม หากข้อมูลเปลี่ยนจะหยุดให้ตรวจ ไม่เขียนทับ split เดิม การตรวจนี้ครอบคลุมภาพซ้ำแบบไฟล์เหมือนกันเท่านั้น ยังต้องตรวจภาพคล้ายกันและกลุ่มผู้เขียนเพิ่มเติม


In [8]:
split_meta = prepare_split(
    data_dir=data_dir,
    audit_dir=audit_dir,
    split_dir=split_dir,
    seed=SEED,
    expected_classes=EXPECTED_CLASSES,
)

# แยกผลของแต่ละ split ออกจากกัน เพื่อไม่ปนคะแนนจากคนละชุดข้อมูล
split_id = split_meta["split_hash"][:12]
experiment_dir = results_dir / MODE / split_id
experiment_dir.mkdir(parents=True, exist_ok=True)

train_rows = read_rows(split_dir / "train.csv")
val_rows = read_rows(split_dir / "val.csv")
print("ภาพ train:", len(train_rows))
print("ภาพ validation:", len(val_rows))
print("ไฟล์ split สำหรับแชร์ให้ทีม:", split_dir)
print("ผลการทดลองจะอยู่ที่:", experiment_dir)


ValueError: Expected 72 classes, found 5. Check label folders.

In [ ]:
train_table = pd.DataFrame(train_rows).assign(split="train")
val_table = pd.DataFrame(val_rows).assign(split="validation")
all_images = pd.concat([train_table, val_table], ignore_index=True)
class_counts = pd.crosstab(all_images["label"], all_images["split"])

display(class_counts)
class_counts.plot.bar(stacked=True, figsize=(16, 4), title="Images per class")
plt.tight_layout()
plt.savefig(experiment_dir / "class_distribution.png", dpi=140)
plt.show()


## 5. ตั้งค่าวิธีฝึกที่ทุกโมเดลใช้ร่วมกัน

ย่อภาพโดยรักษาสัดส่วนแล้วเติมขอบให้เป็นสี่เหลี่ยม ใช้ RGB และ ImageNet mean/std เหมือนกันทุกโมเดล ส่วน augmentation จะค่อยทดลองเพิ่มในขั้นที่ 7

โมเดล pretrained เริ่มจากฝึกหัวจำแนก 2 epochs แล้วจึงปรับทั้งโมเดลด้วย learning rate ที่ต่ำลง ตัวเทรนอยู่ใน [src/train.py](../src/train.py)


In [ ]:
base_config = TrainConfig(
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    epochs=training_budget["benchmark"],
    freeze_epochs=0 if MODE == "smoke" else 2,
    pretrained=MODE != "smoke",
    pad_value=PAD_VALUE,
    device=DEVICE,
    num_workers=0,
    patience=5,
)
save_json(experiment_dir / "base_config.json", asdict(base_config))


## ตรวจสัญญาจากบทเรียน Week 8–11 ก่อนเริ่มเทรน

เอกสารบทเรียนใช้เป็นหลักการประกอบการออกแบบ ไม่ใช่คำสั่งให้เปลี่ยนโจทย์หรือคัดลอกโค้ดทั้งหมด:

| บทเรียน | สิ่งที่ใช้ในโปรเจกต์นี้ | หลักฐานใน pipeline |
|---|---|---|
| Week 8 — Transfer Learning / Data Augmentation | เปลี่ยน classifier ให้มี 72 outputs, ฝึก head ก่อนแล้ว fine-tune ด้วย LR ต่ำกว่า; augmentation ใช้เฉพาะ train และต้องตรวจภาพก่อน | `pretrained`, `freeze_epochs`, `head_lr`, `finetune_lr`, `make_transform(..., training=True)` |
| Week 9 — Multiclass vs Multilabel/Regression | หนึ่งภาพมีหนึ่งคลาส จึงคืน logits รูป `[batch, classes]` และใช้ `CrossEntropyLoss`; softmax ใช้ตอนอ่าน probability เท่านั้น | manifest หนึ่ง label ต่อภาพ + shape/loss checks ด้านล่าง |
| Week 10 — Practical PyTorch CNN | Dataset → DataLoader → model → loss/optimizer → train → validation → checkpoint/inference | `src.train`, `src.search`, `src.inference` |
| Week 11 — Object Detection | ไม่เพิ่ม R-CNN/YOLO เพราะข้อมูลนี้ไม่มี bounding boxes และโจทย์คือ image classification | ระบุเป็น `not_applicable` เพื่อกัน scope creep |

แนวคิด Cutout, Mixup, CutMix, progressive/adaptive augmentation จาก Week 8 ยังไม่เปิดใน benchmark แรก เพราะอาจลบหรือผสมเส้นที่กำหนดเอกลักษณ์ตัวอักษร หากจะใช้ต้องเป็น ablation แยกบน validation split เดิม


In [ ]:
# Executable lesson contract: fail early if task/loss/output/preprocessing disagree.
from torch.utils.data import DataLoader
from src.train import ManifestDataset, build_model, load_split

lesson_train, lesson_val, lesson_mapping, lesson_meta = load_split(data_dir, split_dir)
assert len(lesson_mapping) == EXPECTED_CLASSES
assert all(isinstance(row["label"], str) and row["label"] in lesson_mapping
           for row in lesson_train + lesson_val)

lesson_dataset = ManifestDataset(
    data_dir,
    lesson_train,
    lesson_mapping,
    make_transform(base_config, training=False),
)
lesson_loader = DataLoader(lesson_dataset, batch_size=min(2, len(lesson_dataset)), shuffle=False)
lesson_images, lesson_targets = next(iter(lesson_loader))
assert lesson_targets.dtype == torch.long  # multiclass class indices, not multilabel vectors

probe_architecture = ARCHITECTURES[0] if ARCHITECTURES else "custom_cnn"
probe_config = replace(base_config, architecture=probe_architecture, pretrained=False)
probe_model = build_model(probe_config, len(lesson_mapping), load_pretrained=False).cpu()

# Week 8: frozen feature extractor + trainable replacement classifier.
for parameter in probe_model.parameters():
    parameter.requires_grad_(False)
probe_head = probe_model.get_classifier()
for parameter in probe_head.parameters():
    parameter.requires_grad_(True)
assert sum(p.numel() for p in probe_model.parameters() if p.requires_grad) == sum(
    p.numel() for p in probe_head.parameters()
)

# Week 9/10: raw logits + CrossEntropyLoss + backward; softmax is inference-only.
probe_model.eval()
lesson_logits = probe_model(lesson_images.cpu())
assert lesson_logits.shape == (len(lesson_targets), len(lesson_mapping))
lesson_loss = torch.nn.CrossEntropyLoss()(lesson_logits, lesson_targets.cpu())
lesson_loss.backward()
assert torch.isfinite(lesson_loss) and any(p.grad is not None for p in probe_head.parameters())
lesson_probabilities = lesson_logits.detach().softmax(dim=1)
assert torch.allclose(lesson_probabilities.sum(dim=1), torch.ones(len(lesson_targets)), atol=1e-5)

lesson_contract = {
    "week8_transfer": "head_then_finetune" if base_config.pretrained else "smoke_mechanics_only",
    "week8_augmentation": "train_only_separate_ablation",
    "week9_task": "single_label_multiclass",
    "week9_output_loss": "raw_logits_cross_entropy",
    "week10_pipeline": "dataset_loader_train_validate_checkpoint_inference",
    "week11_detection": "not_applicable_no_bounding_boxes",
    "split_hash": lesson_meta["split_hash"],
}
save_json(experiment_dir / "lesson_contract.json", lesson_contract)
display(pd.Series(lesson_contract, name="decision"))
del probe_model, probe_head, lesson_logits, lesson_loss, lesson_probabilities


### ดูภาพหลังแปลง

แถวบนคือภาพที่โมเดลจะได้รับตามปกติ แถวล่างเพิ่มการหมุนไม่เกิน 5°, เลื่อน 3% และปรับขนาดเล็กน้อย ลองดูว่าเส้นหรือเครื่องหมายสำคัญยังอยู่ครบก่อนใช้ augmentation


In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(13, 5))
normal_transform = make_transform(base_config)
augmented_config = replace(base_config, augmentation=True)
augmented_transform = make_transform(augmented_config, training=True)
for i, row in enumerate(train_rows[:6]):
    with Image.open(data_dir / row["path"]) as source:
        original = source.convert("RGB")
    for j, transform in enumerate([normal_transform, augmented_transform]):
        tensor = transform(original)
        pixels = tensor.permute(1, 2, 0).numpy() * np.array(base_config.std) + np.array(base_config.mean)
        axes[j, i].imshow(pixels.clip(0, 1))
        axes[j, i].set_title(f"{row['label']} / {'aug' if j else 'base'}")
for axis in axes.flat:
    axis.axis("off")
plt.tight_layout()
plt.savefig(experiment_dir / "preprocessing_preview.png", dpi=140)
plt.show()


## 6. เทียบโมเดล

ฝึก Custom CNN จากศูนย์ และลอง pretrained models ตามรายการ `ARCHITECTURES` ด้วยเงื่อนไขเดียวกัน ตารางเรียงจาก macro F1 สูงสุด โดยใช้ accuracy ตัดสินเมื่อ F1 เท่ากัน

ฟังก์ชันคืนตารางผลและตำแหน่ง config ของแต่ละรอบ เราจะนำ config ของอันดับแรกไปทดลองต่อ


In [ ]:
benchmark_results = run_benchmark(
    base=base_config,
    architectures=ARCHITECTURES,
    data_dir=data_dir,
    split_dir=split_dir,
    output_dir=experiment_dir,
)

score_columns = [
    "backbone", "val_macro_f1", "val_accuracy",
    "best_epoch", "parameter_count", "training_seconds",
]
display(benchmark_results[score_columns])

best_backbone_run = benchmark_results.iloc[0].to_dict()
backbone_config = config_from_receipt(best_backbone_run)
print("โมเดลที่เลือกไปทดลองต่อ:", backbone_config.architecture)


### ดูผลด้วยตา: ภาพจริง + เฉลย + คำทำนายของทุกโมเดล

ตารางคะแนนบอกภาพรวม ส่วนนี้แสดง **ภาพสุ่มชุดเดียวกัน** เทียบทุกโมเดลและมีปุ่มเปิดรายละเอียดรายโมเดล
เห็นภาพจริง, label เฉลย, label ที่ทำนาย, confidence และข้อความถูก/ผิดพร้อมสีเขียว/แดง
รายละเอียดมีภาพทายผิดที่มั่นใจสูง, ภาพความมั่นใจต่ำ, recall รายคลาส และคู่คลาสที่สับสน
ภาพทายผิดถูกเลือกมาเพื่อวิเคราะห์ จึงใช้ประมาณ accuracy ทั้งชุดไม่ได้

**Validation / Evaluation / Predict ต่างกันอย่างไร:** train ใช้ปรับ weights; validation ใช้วัดผลและเลือก checkpoint;
evaluation ในส่วนนี้อ่านคำทำนาย validation ทั้งชุดของ best checkpoint; predict โหลด checkpoint มาทำนายภาพทีละภาพแล้วเทียบเฉลยเมื่อมี
validation นี้เคยใช้เลือกโมเดลแล้ว จึงไม่ใช่ unseen test และ Top-3/confidence ไม่ใช่ accuracy ของภาพเดียว

รายงาน HTML ฝังภาพในไฟล์ เปิดดูภายหลังหรือส่งให้ทีมได้ ไม่ต้องเทรนใหม่และไม่ต้องมี GPU
หากชื่อคลาสเป็นรหัส เติม `DISPLAY_LABELS` ด้วย mapping ที่ตรวจแล้ว เช่น `{"class_01": "ก"}`; ไม่เดาตัวอักษรจากเลขคลาส

**ดูผลที่เทรนไว้แล้ว:** รัน cells imports และเตรียมข้อมูล/split เท่านั้น แล้วตั้ง `REVIEW_TABLE` เป็น path ของ
`screening.csv` / `backbone_screening.csv` / `confirmation_runs.csv` ที่ต้องการดู แล้วรัน cell นี้
ต้องใช้ dataset และ split ของรอบนั้น ไฟล์ checkpoint ใน receipt ต้องยังชี้ไปโฟลเดอร์ run เดิม
ตาราง confirmation แสดงแต่ละ seed แยกกัน ไม่เอาผลปนกับ screening


In [ ]:
from IPython.display import HTML
from src.visualize import validation_gallery, live_prediction_gallery, document

REVIEW_TABLE = ""  # ใส่ path CSV เพื่ออ่านผลเดิม โดยไม่รัน training cells
DISPLAY_LABELS = {}  # mapping รหัสคลาส -> อักษรไทย ที่ตรวจแล้ว; ว่าง = แสดง label เดิม
VISUAL_SAMPLE_COUNT = 8
if REVIEW_TABLE:
    review_path = Path(REVIEW_TABLE).expanduser().resolve()
    review_receipts = pd.read_csv(review_path).to_dict("records")
    review_dir = review_path.parent / (review_path.stem + "_visuals")
    review_context = "ผลที่บันทึกไว้: " + str(review_path)
else:
    review_receipts = benchmark_results.to_dict("records")
    review_dir = experiment_dir / "screening_visuals"
    review_context = MODE
if any(r["split_hash"] != split_meta["split_hash"] for r in review_receipts):
    raise ValueError("รายงานกับ split ปัจจุบันไม่ตรงกัน ให้โหลด split ของรอบนั้น")
visual_html = validation_gallery(review_receipts, data_dir, val_rows, review_dir,
    seed=SEED, sample_count=VISUAL_SAMPLE_COUNT, labels=DISPLAY_LABELS, context=review_context)
display(HTML(visual_html))
print("เปิดรายงานได้โดยไม่เทรนใหม่:", review_dir / "index.html")

# ใช้ index ด้านซ้ายเลือกโมเดลสำหรับ Top-3 ในส่วน predict ด้านล่าง
display(pd.DataFrame(review_receipts)[["backbone", "seed", "checkpoint_path"]].reset_index(names="model_index"))

## 7. ลองเพิ่ม augmentation และ class weights

เริ่มจากเทียบเปิด/ปิด augmentation แล้วใช้วิธีที่ชนะไปลอง class weights ต่อ วิธีนี้ช่วยให้เห็นว่าแต่ละอย่างเพิ่มคะแนนหรือไม่

Class weights เพิ่มน้ำหนัก loss ให้คลาสที่มีภาพน้อย คำนวณจาก train เท่านั้น ใช้รากที่สองของอัตราส่วนจำนวนภาพ และจำกัดน้ำหนักก่อน normalize ไม่ให้สูงเกิน 3


In [ ]:
ablation_results = run_ablations(
    base=backbone_config,
    data_dir=data_dir,
    split_dir=split_dir,
    output_dir=experiment_dir,
)
display(ablation_results[["run_id", "val_macro_f1", "val_accuracy", "minimum_per_class_recall"]])

best_ablation_run = ablation_results.iloc[0].to_dict()
selected_config = config_from_receipt(best_ablation_run)
print("ใช้ augmentation:", selected_config.augmentation)
print("ใช้ class weights:", selected_config.class_weights)


## 8. ให้ Optuna ช่วยจูนค่า

เมื่อเลือกโมเดลและวิธีฝึกแล้ว จึงค้นหา learning rate, weight decay และ dropout โดยคงข้อมูลและเงื่อนไขอื่นไว้เหมือนเดิม Trial ที่แนวโน้มไม่ดีอาจถูกหยุดก่อนเพื่อลดเวลา ค่าที่ไม่ได้ใช้กับโมเดลนั้นจะไม่ถูกนำมาจูน

ผลเก็บใน `optuna.sqlite3` กดรันซ้ำจะทำต่อให้ครบจำนวน trials ที่ตั้งไว้ งานที่เสร็จแล้วใช้ซ้ำได้ แต่งานที่ถูกขัดจังหวะจะเริ่มฝึกใหม่ตั้งแต่ epoch แรก


In [ ]:
tuning_config = replace(selected_config, epochs=training_budget["hpo_epochs"])
study = tune(
    base=tuning_config,
    data_dir=data_dir,
    split_dir=split_dir,
    output_dir=experiment_dir,
    n_trials=training_budget["trials"],
)

display(study.trials_dataframe())
print("Trial ที่ได้คะแนนสูงสุด:", study.best_trial.number)
print("Macro F1:", study.best_value)
display(pd.Series(study.best_params, name="ค่าที่ Optuna เลือก"))


## 9. เทรนยืนยันก่อนเลือก config สุดท้าย

นำ 2 config ที่ดีที่สุดจาก Optuna มาเทรนให้นานขึ้น พร้อม config ก่อนจูนไว้เทียบ โหมด `full` ฝึกซ้ำด้วย seed 42 และ 123 แล้วเลือกจากค่าเฉลี่ย macro F1

ขั้นนี้สร้าง `leaderboard.csv` แยกจากผลการทดลองสั้น คะแนนยังเป็น validation ที่ใช้จูน จึงต้องวัดกับ unseen test อีกครั้งก่อนสรุปผลทั่วไป


In [ ]:
leaderboard, best_run = confirm(
    base=tuning_config,
    study=study,
    data_dir=data_dir,
    split_dir=split_dir,
    output_dir=experiment_dir,
    epochs=training_budget["final_epochs"],
    seeds=training_budget["seeds"],
    top_k=2,
)
display(leaderboard)


### เก็บไฟล์สำหรับใช้งานต่อ

เลือก config จากคะแนนเฉลี่ย แต่ใช้ checkpoint ของ seed แรกที่กำหนดไว้ เพื่อไม่เลือกเฉพาะ seed ที่บังเอิญคะแนนสูง ส่งโฟลเดอร์ `export` นี้ให้เพื่อนได้


In [ ]:
export_dir = experiment_dir / "export"
export_dir.mkdir(exist_ok=True)
shutil.copy2(best_run["checkpoint_path"], export_dir / "best_model.pt")
shutil.copy2(experiment_dir / "best_config.json", export_dir / "best_config.json")
shutil.copy2(split_dir / "label_to_index.json", export_dir / "label_to_index.json")
shutil.copy2(split_dir / "split_meta.json", export_dir / "split_meta.json")
print("ไฟล์ config:", export_dir / "best_config.json")
print("ไฟล์โมเดล:", export_dir / "best_model.pt")
print("Config ที่เลือก:", best_run["candidate"], "seed ที่ใช้:", best_run["seed"])


## 10. ดูผลและข้อผิดพลาด

กราฟช่วยดูว่าการฝึกดีขึ้นเมื่อไร และเริ่ม overfit หรือไม่ ค่า train วัดขณะเปิด dropout/augmentation จึงอาจต่างจาก validation ได้

ตารางทำนายและ confusion matrix มาจาก epoch ที่เลือกเก็บ checkpoint


In [ ]:
best_run_dir = Path(best_run["checkpoint_path"]).parent
history = pd.read_csv(best_run_dir / "history.csv")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
history.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0])
history.plot(x="epoch", y=["train_accuracy", "val_accuracy"], ax=axes[1])
history.plot(x="epoch", y="val_macro_f1", ax=axes[2])
plt.tight_layout()
plt.savefig(experiment_dir / "learning_curves.png", dpi=140)
plt.show()


### คลาสไหนสับสนกัน

แกนตั้งคือคลาสจริง แกนนอนคือคลาสที่ทำนาย เลขบนแกนเรียงตาม `label_to_index.json`


In [ ]:
confusion_table = pd.read_csv(best_run_dir / "confusion_matrix.csv", index_col=0)
fig, ax = plt.subplots(figsize=(10, 9))
plot = ax.imshow(confusion_table.to_numpy(), cmap="Blues")
fig.colorbar(plot, ax=ax)
ax.set(xlabel="Predicted class index", ylabel="True class index", title="Best checkpoint confusion matrix")
plt.tight_layout()
plt.savefig(experiment_dir / "confusion_matrix.png", dpi=140)
plt.show()


### ภาพที่ทายผิด

เริ่มดูภาพที่ทายผิดแต่มั่นใจสูง แล้วพิจารณาว่าเกิดจากภาพเบลอ เส้นขาด คลาสคล้ายกัน หรือ label ผิด ใช้ตัวอย่างเหล่านี้กำหนดการทดลองรอบถัดไป


In [ ]:
predictions = pd.read_csv(best_run_dir / "predictions.csv", dtype={"true_label": str, "predicted_label": str})
errors = predictions[predictions.true_label != predictions.predicted_label].sort_values("confidence", ascending=False)
errors.to_csv(experiment_dir / "errors.csv", index=False)
confused_pairs = errors.groupby(["true_label", "predicted_label"]).size().sort_values(ascending=False).rename("count")
display(confused_pairs.head(15).to_frame())
if not errors.empty:
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for axis, (_, row) in zip(axes.flat, errors.head(8).iterrows()):
        with Image.open(data_dir / row.path) as source:
            axis.imshow(source.convert("RGB"))
        axis.set_title(f"{row.true_label} → {row.predicted_label} ({row.confidence:.2f})")
    for axis in axes.flat:
        axis.axis("off")
    plt.tight_layout()
    plt.savefig(experiment_dir / "error_examples.png", dpi=140)
    plt.show()
else:
    print("ไม่พบภาพที่ทายผิดใน validation รอบนี้")


## 11. ลองทำนายภาพ

โหลดโมเดลจากไฟล์ที่ export ไว้ พร้อม preprocessing และรายชื่อคลาสเดิม เปลี่ยน `sample_image` เป็น path ของภาพใหม่ที่ต้องการลองได้


In [ ]:
sample_image = data_dir / val_rows[0]["path"]
predictor = Predictor(export_dir / "best_model.pt", device=DEVICE)
result = predictor.predict([sample_image], top_k=3)
display(pd.DataFrame(result[0]["top_k"]))
plt.figure(figsize=(3, 3))
with Image.open(sample_image) as source:
    plt.imshow(source.convert("RGB"))
plt.title(result[0]["predicted_label"])
plt.axis("off")
plt.show()


### เช็กว่าไฟล์โมเดลโหลดกลับมาใช้ได้ตรงกัน

ทำนาย validation อีกรอบด้วยโมเดลที่โหลดใหม่ ผลควรตรงกับตารางที่บันทึกตอนฝึก


In [ ]:
validation_paths = [data_dir / row["path"] for row in val_rows]
reloaded_predictions = predictor.predict(validation_paths, batch_size=BATCH_SIZE)
reloaded_labels = [result["predicted_label"] for result in reloaded_predictions]
saved_labels = predictions["predicted_label"].tolist()

assert reloaded_labels == saved_labels, "ผลจาก checkpoint ไม่ตรงกับผลที่บันทึกไว้"
print("โหลดโมเดลกลับมาแล้วทำนายตรงกัน")
print("ผลทั้งหมด:", experiment_dir)
if MODE == "smoke":
    print("รอบนี้ใช้ภาพจำลองเพื่อทดสอบระบบ ยังไม่ใช่ผลทดลองข้อมูลจริง")


### เปิดดูภาพของโมเดลสุดท้าย และลองภาพของเราเอง

ส่วนแรกเป็น evaluation report ของ checkpoint ที่ export จริง ส่วนที่สองเรียก `Predictor` จริงบนภาพที่เลือก
แสดงภาพต้นฉบับ → ภาพที่โมเดลเห็นหลัง resize/pad (แปลง normalization กลับเพื่อให้ดูได้) → Top-3 → เทียบเฉลย
ภาพ validation ใช้เฉลยจาก manifest อัตโนมัติ; ภาพนอกชุดต้องใส่ `IMAGE_TRUTH` เองตาม label mapping
ไม่ตั้งเฉลยจากคำทำนายของโมเดล และไม่เอาภาพสาธิตมารวมใน validation metrics


ตั้ง `PREDICT_MODEL_INDEX` ตาม model_index ในตาราง screening เพื่อดู Top-3 ของโมเดลอื่น; `None` = โมเดลสุดท้ายที่ export

In [ ]:
final_visual_dir = experiment_dir / "final_visuals"
display(HTML(validation_gallery([best_run], data_dir, val_rows, final_visual_dir,
    seed=SEED, sample_count=VISUAL_SAMPLE_COUNT, labels=DISPLAY_LABELS, context=MODE)))

PREDICT_MODEL_INDEX = None  # None = exported model; 0, 1, ... = โมเดลจากตาราง review
if PREDICT_MODEL_INDEX is None:
    live_predictor = predictor
    live_context = MODE
else:
    if not 0 <= PREDICT_MODEL_INDEX < len(review_receipts):
        raise ValueError("model_index ไม่อยู่ในตาราง review")
    live_predictor = Predictor(Path(review_receipts[PREDICT_MODEL_INDEX]["checkpoint_path"]), device=DEVICE)
    live_context = review_context

PREDICT_IMAGES = []  # เช่น ["/absolute/path/photo.png"]; ว่าง = สุ่ม validation ไม่เกิน 8 รูป
IMAGE_TRUTH = {}  # เช่น {"/absolute/path/photo.png": "class_01"}; ไม่มีเฉลยจะไม่ตัดสินถูก/ผิด
if PREDICT_IMAGES:
    predict_paths = [Path(p).expanduser().resolve() for p in PREDICT_IMAGES]
else:
    samples = pd.DataFrame(val_rows).sort_values("path").sample(n=min(8, len(val_rows)), random_state=SEED)
    predict_paths = [data_dir / p for p in samples.path]
known_truth = {str((data_dir / r["path"]).resolve()): r["label"] for r in val_rows}
known_truth.update({str(Path(p).expanduser().resolve()): str(label) for p, label in IMAGE_TRUTH.items()})
valid_labels = set(live_predictor.labels)
if any(label not in valid_labels for label in known_truth.values()):
    raise ValueError("เฉลยต้องใช้ label ที่มีใน label_to_index.json")
live_html, live_answers = live_prediction_gallery(live_predictor, predict_paths,
    truth=known_truth, labels=DISPLAY_LABELS, context=live_context)
display(HTML(live_html))
display(live_answers.drop(columns="top3"))
live_answers.to_csv(final_visual_dir / "live_predictions.csv", index=False)
(final_visual_dir / "live_predictions.html").write_text(document("Live predictions", live_html), encoding="utf-8")
print("รายงานภาพโมเดลสุดท้าย:", final_visual_dir)


## ทดลองต่อและส่งผลให้ทีม

เพิ่มโมเดลใน `ARCHITECTURES` แล้วรันด้วย split และเงื่อนไขเดิม เวลาส่งผลให้แนบ `leaderboard.csv`, `confirmation_runs.csv` และโฟลเดอร์ `export` อย่าเทียบคะแนนข้ามโหมด smoke/quick/full

ผลไม่ได้เก็บใน Git ถ้าใช้ Colab ให้เก็บ output และ split ไว้บน Drive ก่อนปิด session และหลีกเลี่ยงการเปิดหลาย notebook เขียน output เดียวกัน

วิธีรัน config เดียวและคำอธิบายไฟล์อยู่ใน [README](README.md) ส่วนลำดับการค้นหาอยู่ใน [src/search.py](../src/search.py)

เอกสารไลบรารี: [timm](https://huggingface.co/docs/timm/main/quickstart) · [Optuna](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.create_study.html)
